# Post-process: DB1 → DB.F

Takes every unprocessed sighting in **DB1** (`detections`) and turns it into clean potholes in **DB.F** (`potholes`):

1. **Locate** — move the point from where the phone was to where the pothole is (flat-road geometry).
2. **Merge** — sightings within a few metres of each other are the same pothole.
3. **Score** — a local vision SLM (via Ollama) gives severity, ride-impact score and a description. If it fails, a simple size rule is used and the pothole is flagged for review.

Run it by hand (Run All), or from the server with `POST /process` (papermill). The next cell holds the parameters papermill can override.

In [1]:
# Parameters (papermill overrides these)
DB_PATH = None            # None -> server default (data/pothole.db)
PHOTOS_DIR = None         # None -> server default (data/photos)
MERGE_RADIUS_M = 5.0      # sightings closer than this are one pothole
MAX_MERGE_RADIUS_M = 10.0 # radius grows with poor GPS, but never past this
MAX_DIST_M = 25.0         # ignore detections farther ahead than this (too inaccurate)
MAX_GPS_ACCURACY_M = 30.0 # ignore frames with worse GPS than this
MIN_SIGHTINGS = 3         # shown to drivers only after this many sightings
USE_SLM = True            # False -> size rule only
OLLAMA_URL = None         # None -> server default
SLM_MODEL = None          # None -> server default (qwen3-vl:4b)
COMPARE_MODELS = []       # e.g. ['qwen3-vl:4b', 'gemma4:e4b'] to compare models (writes nothing)
LIMIT = None              # max detections per run, None means all

## Setup

In [2]:
import base64, io, json, math, sys
from pathlib import Path
from datetime import datetime, timezone

import httpx
import pandas as pd
from PIL import Image

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
from backend import db, geo
from backend.config import settings

DB_PATH = Path(DB_PATH) if DB_PATH else settings.DB_PATH
PHOTOS_DIR = Path(PHOTOS_DIR) if PHOTOS_DIR else settings.PHOTOS_DIR
OLLAMA_URL = OLLAMA_URL or settings.OLLAMA_URL
SLM_MODEL = SLM_MODEL or settings.SLM_MODEL
CROPS_DIR = PHOTOS_DIR / 'crops'
CROPS_DIR.mkdir(parents=True, exist_ok=True)

db.init(DB_PATH)
conn = db.connect(DB_PATH)
now = db.now_iso()
print('DB:', DB_PATH, '| photos:', PHOTOS_DIR, '| SLM:', SLM_MODEL if USE_SLM else 'off')

DB: /Users/msaisanjeev/Downloads/RoadPlotter 3/data/pothole.db | photos: /Users/msaisanjeev/Downloads/RoadPlotter 3/data/photos | SLM: qwen3.5:4b-q4_K_M


## Load unprocessed sightings

In [3]:
q = 'SELECT * FROM detections WHERE processed = 0 ORDER BY id'
if LIMIT: q += f' LIMIT {int(LIMIT)}'
dets = [dict(r) for r in conn.execute(q)]
print(f'{len(dets)} unprocessed detections')
pd.DataFrame(dets)[['id','lat','lon','heading_deg','confidence','gps_accuracy_m']].head(10) if dets else None

195 unprocessed detections


,id,lat,lon,heading_deg,confidence,gps_accuracy_m
0,1,12.372476,76.584727,206.0,0.321,3.79
1,2,12.372395,76.584685,208.0,0.307,3.79
2,3,12.372222,76.584596,206.0,0.368,3.79
3,4,12.372222,76.584596,206.0,0.644,3.79
4,5,12.372222,76.584596,206.0,0.467,3.79
5,6,12.372192,76.584580,207.0,0.364,3.79
6,7,12.372131,76.584547,207.0,0.412,3.79
7,8,12.371720,76.584328,206.0,0.453,3.79
8,9,12.371493,76.584212,206.0,0.800,3.79
9,10,12.371493,76.584212,206.0,0.377,3.79


## Step 1 — Locate each pothole

Bottom-centre of the box → distance ahead and sideways on a flat road, using the mount height, tilt and field of view sent with the frame. Then shift the phone's lat/long by that offset along its heading.

In [4]:
from collections import Counter
from backend.postprocess_quality import locate
# Include previously accepted observations for conservative GPS heading recovery.
observations = [dict(r) for r in conn.execute('SELECT * FROM detections ORDER BY id')]
located, rejected = [], []
for d in dets:
    item, reason = locate(d, observations, MAX_DIST_M, MAX_GPS_ACCURACY_M)
    if reason:
        rejected.append((d['id'], reason))
    else:
        located.append(item)
rejection_counts = dict(Counter(reason for _, reason in rejected))
print('Accepted:', len(located), '| Deferred:', len(rejected))
print('Deferred reasons:', rejection_counts)
print('GPS headings recovered:', sum(d['heading_recovered'] for d in located))
pd.DataFrame(located)[['id','forward_m','p_lat','p_lon','heading_recovered']].head(10) if located else None


Accepted: 115 | Deferred: 80
Deferred reasons: {'unprojectable_box': 76, 'too_far': 4}
GPS headings recovered: 0


,id,forward_m,p_lat,p_lon,heading_recovered
0,8,0.844628,12.371711,76.584329,False
1,12,1.408051,12.370759,76.583843,False
2,13,5.353692,12.370718,76.583846,False
3,15,4.050467,12.370399,76.583687,False
4,16,0.801099,12.370431,76.583688,False
5,17,3.547587,12.369181,76.582674,False
6,18,1.591928,12.369177,76.582649,False
7,19,1.636610,12.369158,76.582618,False
8,20,4.706143,12.369132,76.582603,False
9,21,3.143680,12.369135,76.582574,False


## Step 2 — Merge into potholes

Each located sighting joins the nearest open pothole within its merge radius (already in DB.F, or created earlier in this run). The radius grows with poor GPS, up to `MAX_MERGE_RADIUS_M`. Positions are a weighted average, so more sightings → better location. A final pass joins potholes whose averaged positions ended up closer than `MERGE_RADIUS_M`.

In [5]:
def open_potholes_near(lat, lon, radius):
    box = geo.bbox_around(lat, lon, radius)
    return db.potholes_in_box(conn, *box, statuses=('open', 'unconfirmed'))

def merge_radius(d):
    return min(MAX_MERGE_RADIUS_M, max(MERGE_RADIUS_M, 1.5 * (d['gps_accuracy_m'] or 5.0)))

assign = {}        # detection id -> pothole id
touched = set()
for d in sorted(located, key=lambda d: -d['weight']):   # best sightings anchor the clusters
    r = merge_radius(d)
    cands = [p for p in open_potholes_near(d['p_lat'], d['p_lon'], r) if p['damage_type'] == d['damage_type']]
    best = min(cands, key=lambda p: geo.haversine_m(p['lat'], p['lon'], d['p_lat'], d['p_lon']), default=None)
    if best and geo.haversine_m(best['lat'], best['lon'], d['p_lat'], d['p_lon']) <= r:
        ws = best['weight_sum'] + d['weight']
        lat = (best['lat'] * best['weight_sum'] + d['p_lat'] * d['weight']) / ws
        lon = (best['lon'] * best['weight_sum'] + d['p_lon'] * d['weight']) / ws
        conn.execute('''UPDATE potholes SET lat=?, lon=?, weight_sum=?, times_seen=times_seen+1,
                        last_seen=?, updated_at=? WHERE id=?''',
                     (lat, lon, ws, d['captured_at'] or now, now, best['id']))
        pid = best['id']
    else:
        cur = conn.execute('''INSERT INTO potholes (lat, lon, weight_sum, times_seen, first_seen, last_seen, status, updated_at)
                              VALUES (?,?,?,?,?,?, 'unconfirmed', ?)''',
                           (d['p_lat'], d['p_lon'], d['weight'], 1, d['captured_at'] or now, d['captured_at'] or now, now))
        pid = cur.lastrowid
        conn.execute('UPDATE potholes SET damage_type=? WHERE id=?', (d['damage_type'],pid))
    assign[d['id']] = pid
    touched.add(pid)

# consolidation: join potholes whose averaged positions are now close together
def join(keep, lose):
    a = dict(conn.execute('SELECT * FROM potholes WHERE id=?', (keep,)).fetchone())
    b = dict(conn.execute('SELECT * FROM potholes WHERE id=?', (lose,)).fetchone())
    ws = a['weight_sum'] + b['weight_sum']
    conn.execute('''UPDATE potholes SET lat=?, lon=?, weight_sum=?, times_seen=?, first_seen=?, last_seen=?, updated_at=?
                    WHERE id=?''',
                 ((a['lat'] * a['weight_sum'] + b['lat'] * b['weight_sum']) / ws,
                  (a['lon'] * a['weight_sum'] + b['lon'] * b['weight_sum']) / ws, ws,
                  a['times_seen'] + b['times_seen'], min(a['first_seen'], b['first_seen']),
                  max(a['last_seen'], b['last_seen']), now, keep))
    conn.execute('UPDATE detections SET pothole_id=? WHERE pothole_id=?', (keep, lose))
    conn.execute('UPDATE feedback SET pothole_id=? WHERE pothole_id=?', (keep, lose))
    conn.execute('DELETE FROM potholes WHERE id=?', (lose,))
    for k, v in assign.items():
        if v == lose: assign[k] = keep
    touched.discard(lose); touched.add(keep)

changed = True
while changed:
    changed = False
    for pid in sorted(touched):
        p = conn.execute('SELECT lat, lon, weight_sum, damage_type FROM potholes WHERE id=?', (pid,)).fetchone()
        if p is None: continue
        near = [q for q in open_potholes_near(p['lat'], p['lon'], MERGE_RADIUS_M) if q['id'] != pid and q['damage_type'] == p['damage_type']
                and geo.haversine_m(p['lat'], p['lon'], q['lat'], q['lon']) <= MERGE_RADIUS_M]
        if near:
            q = min(near, key=lambda q: geo.haversine_m(p['lat'], p['lon'], q['lat'], q['lon']))
            keep, lose = (pid, q['id']) if p['weight_sum'] >= q['weight_sum'] else (q['id'], pid)
            join(keep, lose); changed = True
            break

# location accuracy: spread of the sightings around the merged point
for pid in touched:
    p = dict(conn.execute('SELECT lat, lon FROM potholes WHERE id=?', (pid,)).fetchone())
    pts = [d for d in located if assign[d['id']] == pid]
    prev = conn.execute('SELECT lat, lon, gps_accuracy_m FROM detections WHERE pothole_id=?', (pid,)).fetchall()
    spread = [geo.haversine_m(p['lat'], p['lon'], d['p_lat'], d['p_lon']) for d in pts]
    gps = [d['gps_accuracy_m'] or 10 for d in pts] + [r['gps_accuracy_m'] or 10 for r in prev]
    n = len(gps)
    acc = max(sum(spread) / max(1, len(spread)), (sum(gps) / n) / math.sqrt(n))
    conn.execute('UPDATE potholes SET loc_accuracy_m=? WHERE id=?', (round(acc, 1), pid))

# only potholes seen often enough are shown to drivers
for pid in touched:
    conn.execute('''UPDATE potholes SET status = CASE WHEN times_seen >= ? THEN 'open' ELSE 'unconfirmed' END
                    WHERE id=? AND status IN ('open', 'unconfirmed')''', (MIN_SIGHTINGS, pid))
conn.commit()
st = dict(conn.execute(f"SELECT status, COUNT(*) FROM potholes WHERE id IN ({','.join(map(str, touched)) or '0'}) GROUP BY status").fetchall())
print(f'{len(touched)} potholes created or updated: {st}')

97 potholes created or updated: {'open': 2, 'unconfirmed': 95}


## Step 3 — Severity, ride-impact score, description (local SLM)

For each touched pothole: pick the best sighting (confident, 5–15 m away), crop around the pothole, and ask the SLM. It gets the crop plus the measured numbers and must answer in JSON. Bad or missing answers fall back to a size rule and are flagged `needs_review`.

In [6]:
# Finalize assignments before optional scoring; a missing image must not duplicate sightings.
for detection_id, pothole_id in assign.items():
    conn.execute('UPDATE detections SET processed=1, pothole_id=?, process_note=NULL WHERE id=?', (pothole_id,detection_id))
conn.commit()
SEVERITIES = ['Low', 'Medium', 'High', 'Critical']

PROMPT = '''You are a road inspector. The photo shows {damage_type}, cropped from a dashcam frame. Do not describe cracks as holes. Physical depth cannot be measured from this photo.
Measured from the camera geometry (approximate):
- width: {width:.2f} m, along-road length: {depth:.2f} m, area: {area:.2f} m2
- position: {lane}
- seen {seen} time(s)

Judge it and answer ONLY with JSON, no other text:
{{"severity": "Low|Medium|High|Critical",
  "ride_score": <number 0-10, how hard it jolts a car or motorbike>,
  "surface": "<short: depth cues, water, broken edges, gravel>",
  "description": "<one plain sentence for drivers, under 20 words, include approx size and position>"}}

Scoring guide:
- Low (0-3): shallow, small (< 30 cm), barely felt.
- Medium (3-5.5): noticeable jolt, 30-60 cm or a bit deep.
- High (5.5-8): deep or wide (> 60 cm), in a wheel path, uncomfortable even slow.
- Critical (8-10): very deep or large, water-filled, can damage tyres or throw a bike.
'''

def best_sighting(pid):
    rows = [dict(r) for r in conn.execute('SELECT * FROM detections WHERE pothole_id=?', (pid,))]
    rows += [d for d in located if assign.get(d['id']) == pid]
    def score(d):
        g = d if 'forward_m' in d else geo.box_to_ground((d['x1'], d['y1'], d['x2'], d['y2']), d['img_w'], d['img_h'],
                                                         d['camera_height_m'], d['camera_tilt_deg'], d['hfov_deg']) or {'forward_m': 99}
        return d['confidence'] - abs(g['forward_m'] - 10) / 30
    d = max(rows, key=score)
    if 'forward_m' not in d:
        d = {**d, **geo.box_to_ground((d['x1'], d['y1'], d['x2'], d['y2']), d['img_w'], d['img_h'],
                                     d['camera_height_m'], d['camera_tilt_deg'], d['hfov_deg'])}
    return d, len(rows)

def make_crop(d, pid):
    img = Image.open(PHOTOS_DIR / d['photo_path']).convert('RGB')
    bw, bh = d['x2'] - d['x1'], d['y2'] - d['y1']
    pad_x, pad_y = max(bw * 0.8, 40), max(bh * 1.2, 40)
    box = (max(0, d['x1'] - pad_x), max(0, d['y1'] - pad_y), min(img.width, d['x2'] + pad_x), min(img.height, d['y2'] + pad_y))
    crop = img.crop(tuple(int(v) for v in box))
    rel = Path('crops') / f'pothole_{pid}.jpg'
    crop.save(PHOTOS_DIR / rel, quality=88)
    return crop, str(rel)

def rule_score(g):
    if g.get('damage_type','pothole') != 'pothole':
        return {'severity': None, 'ride_score': None, 'surface': None,
                'description': g['damage_type'].replace('_',' ').capitalize() + ' detected; severity requires review.'}
    a = g['area_m2']
    sev = 'Low' if a < 0.08 else 'Medium' if a < 0.25 else 'High' if a < 0.7 else 'Critical'
    score = min(10, 1.5 + a * 9 + (1.0 if 'wheel path' in geo.lane_position(g['lateral_m']) else 0))
    lane = geo.lane_position(g['lateral_m'])
    return {'severity': sev, 'ride_score': round(score, 1), 'surface': None,
            'description': f"Pothole about {g['width_m']*100:.0f} cm wide, {lane}."}

def ask_slm(crop, g, seen, model=None):
    buf = io.BytesIO(); crop.save(buf, format='JPEG', quality=88)
    prompt = PROMPT.format(damage_type=g.get('damage_type','pothole').replace('_',' '), width=g['width_m'], depth=g['depth_m'], area=g['area_m2'],
                           lane=geo.lane_position(g['lateral_m']), seen=seen)
    body = {'model': model or SLM_MODEL, 'prompt': prompt, 'stream': False, 'format': 'json',
            'images': [base64.b64encode(buf.getvalue()).decode()], 'options': {'temperature': 0},
            'think': False}   # Qwen3-VL: answer directly, no long reasoning
    r = httpx.post(f'{OLLAMA_URL}/api/generate', timeout=180, json=body)
    if r.status_code == 400 and 'think' in r.text:   # model without a thinking switch
        body.pop('think'); r = httpx.post(f'{OLLAMA_URL}/api/generate', timeout=180, json=body)
    r.raise_for_status()
    out = json.loads(r.json()['response'])
    sev = str(out.get('severity', '')).strip().capitalize()
    score = float(out.get('ride_score'))
    desc = str(out.get('description', '')).strip()
    if sev not in SEVERITIES or not (0 <= score <= 10) or not desc:
        raise ValueError(f'invalid SLM answer: {out}')
    return {'severity': sev, 'ride_score': round(score, 1),
            'surface': (str(out['surface'])[:120] if out.get('surface') else None), 'description': desc[:200]}

slm_ok = USE_SLM
if USE_SLM:
    try:
        httpx.get(f'{OLLAMA_URL}/api/tags', timeout=3).raise_for_status()
    except Exception as e:
        print(f'Ollama not reachable at {OLLAMA_URL} ({e}); using the size rule for this run.')
        slm_ok = False

results = []
for pid in sorted(touched):
    try:
        d, seen = best_sighting(pid)
        crop, crop_rel = make_crop(d, pid)
    except (OSError, ValueError, TypeError) as error:
        conn.execute('UPDATE potholes SET needs_review=1 WHERE id=?', (pid,))
        print(f'Defect {pid}: image/geometry unavailable; kept location, requires review: {error}')
        continue
    scored_by, review, model = 'rule', 1, None
    try:
        if not slm_ok: raise RuntimeError('SLM off')
        s = ask_slm(crop, d, seen); scored_by, review, model = 'slm', 0, SLM_MODEL
    except Exception as e:
        if slm_ok: print(f'pothole {pid}: SLM failed ({e}), using rule')
        s = rule_score(d)
    conn.execute('''UPDATE potholes SET severity=?, ride_score=?, surface=?, description=?, width_m=?, area_m2=?,
                    lane_position=?, best_photo_path=?, crop_path=?, scored_by=?, slm_model=?, needs_review=?, updated_at=?
                    WHERE id=?''',
                 (s['severity'], s['ride_score'], s['surface'], s['description'], round(d['width_m'], 2),
                  round(d['area_m2'], 3), geo.lane_position(d['lateral_m']), d['photo_path'], crop_rel,
                  scored_by, model, review, now, pid))
    results.append({'pothole': pid, **s, 'by': scored_by, 'seen': seen})
conn.commit()
pd.DataFrame(results)

,pothole,severity,ride_score,surface,description,by,seen
0,1,Low,2.0,gravel,"Small gravel patch in lane centre, approx 18cm...",slm,4
1,2,Low,2.0,flat asphalt with minor debris,"Small flat patch on right wheel path, about 30...",slm,2
2,3,Low,2.0,"broken edge, gravel","Small pothole on left wheel path, approx 20cm ...",slm,2
3,4,Low,2.0,gravel,"Small gravel patch in lane centre, approx 9cm ...",slm,2
4,5,Medium,4.0,"broken edges, gravel",A medium-sized pothole is located in the left ...,slm,4
...,...,...,...,...,...,...,...
92,93,Low,2.0,"cracked asphalt, dry","A small longitudinal crack in the lane centre,...",slm,2
93,94,Low,2.0,"fine gravel texture, no water",A small longitudinal crack is located on the l...,slm,2
94,95,Medium,4.0,"cracked asphalt, no water",A 1.11m long crack runs along the left wheel p...,slm,2
95,96,Low,2.0,cracked asphalt,A small longitudinal crack is located to the l...,slm,2


## Mark sightings as processed

In [7]:
for det_id, pid in assign.items():
    conn.execute('UPDATE detections SET processed=1, pothole_id=?, process_note=NULL WHERE id=?', (pid, det_id))
for det_id, reason in rejected:
    conn.execute('UPDATE detections SET processed=0, process_note=? WHERE id=?', (reason, det_id))
conn.commit()

summary = {
    'detections_processed': len(dets),
    'located': len(located),
    'deferred': len(rejected),
    'deferred_reasons': rejection_counts,
    'headings_recovered': sum(d['heading_recovered'] for d in located),
    'potholes_touched': len(touched),
    'open_potholes_total': conn.execute("SELECT COUNT(*) FROM potholes WHERE status='open'").fetchone()[0],
    'scored_by_slm': sum(r['by'] == 'slm' for r in results),
}
conn.close()
summary

{'detections_processed': 195,
 'located': 115,
 'deferred': 80,
 'deferred_reasons': {'unprojectable_box': 76, 'too_far': 4},
 'headings_recovered': 0,
 'potholes_touched': 97,
 'open_potholes_total': 2,
 'scored_by_slm': 97}

## Optional: compare SLMs

Set `COMPARE_MODELS` in the parameters cell (pull each with `ollama pull <name>` first). Every model scores the same open potholes (up to 50) from their saved crops, and the table shows the answers side by side with the time each took. Nothing is written to the database: pick the model whose answers match your own judgement, then set `POTHOLE_SLM_MODEL` to it.

In [8]:
import time
if COMPARE_MODELS:
    conn = db.connect(DB_PATH)
    rows = [dict(r) for r in conn.execute(
        "SELECT * FROM potholes WHERE status='open' AND crop_path IS NOT NULL ORDER BY id LIMIT 50")]
    table = []
    for p in rows:
        g = {'width_m': p['width_m'] or 0.3, 'area_m2': p['area_m2'] or 0.07,
             'depth_m': (p['area_m2'] or 0.07) / max(0.05, (p['width_m'] or 0.3) * math.pi / 4),
             'lateral_m': {'left of lane': -2, 'left wheel path': -0.8, 'lane centre': 0,
                           'right wheel path': 0.8, 'right of lane': 2}.get(p['lane_position'], 0)}
        crop = Image.open(PHOTOS_DIR / p['crop_path']).convert('RGB')
        row = {'pothole': p['id'], 'width_m': p['width_m']}
        for m in COMPARE_MODELS:
            t0 = time.perf_counter()
            try:
                s = ask_slm(crop, g, p['times_seen'], model=m)
                row[m] = f"{s['severity']} {s['ride_score']}: {s['description']}"
            except Exception as e:
                row[m] = f'failed: {e}'[:80]
            row[f'{m} s'] = round(time.perf_counter() - t0, 1)
        table.append(row)
    conn.close()
    pd.set_option('display.max_colwidth', 90)
    display(pd.DataFrame(table))
else:
    print('COMPARE_MODELS is empty: nothing to compare.')

COMPARE_MODELS is empty: nothing to compare.
